In [ ]:
# =========================================================
# ENRICH DC DATA USING "Formatted Address"
# Adds:
#   - census_tract_geocoder
#   - census_block
#   - block_group
#   - historic_year_final
#   - building_age
#
# INPUT FILES:
#   - final_merged_5_geocoded.csv
#   - final_merged_24_geocoded.csv
#
# Uses:
#   - U.S. Census batch address geocoder (geographies)
#   - DC HistoryQuest / Historic Building Info layer
# =========================================================

# pip install pandas requests

import csv
import io
import os
import re
import time
import math
import tempfile
import warnings
import requests
import pandas as pd
import urllib3
import os
import time
import tempfile
import requests
import pandas as pd
import geopandas as gpd

WARD_LAYER = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Administrative_Other_Boundaries_WebMercator/MapServer/53"

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS request.*")

# ---------------------------------------------------------
# INPUT FILES
# ---------------------------------------------------------
INPUT_FILES = [
    "final_merged_5_geocoded.csv",
    "final_merged_24_geocoded.csv",
]

# ---------------------------------------------------------
# ENDPOINTS
# ---------------------------------------------------------
CENSUS_BATCH_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
HISTORIC_LAYER = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Historic_WebMercator/MapServer/10/query"

# ---------------------------------------------------------
# ADDRESS CLEANING
# ---------------------------------------------------------
def clean_whitespace(text):
    if pd.isna(text):
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def strip_country(text):
    if pd.isna(text):
        return None
    s = str(text).strip()
    s = re.sub(r",\s*USA\s*$", "", s, flags=re.I)
    s = re.sub(r",\s*UNITED STATES\s*$", "", s, flags=re.I)
    return s.strip()

def esri_geojson_query_url(layer_url, out_fields="*", where="1=1", result_offset=0, result_record_count=1000):
    return (
        f"{layer_url}/query"
        f"?where={requests.utils.quote(where, safe='=>< ()')}"
        f"&outFields={requests.utils.quote(out_fields, safe=',*')}"
        f"&returnGeometry=true"
        f"&f=geojson"
        f"&outSR=4326"
        f"&resultOffset={result_offset}"
        f"&resultRecordCount={result_record_count}"
    )

def download_geojson_layer(layer_url, out_fields="*", where="1=1", chunk_size=1000):
    all_parts = []
    offset = 0

    while True:
        url = esri_geojson_query_url(
            layer_url=layer_url,
            out_fields=out_fields,
            where=where,
            result_offset=offset,
            result_record_count=chunk_size
        )

        r = requests.get(url, timeout=120, verify=False)
        r.raise_for_status()

        with tempfile.NamedTemporaryFile(delete=False, suffix=".geojson") as tmp:
            tmp.write(r.content)
            tmp_path = tmp.name

        gdf = gpd.read_file(tmp_path)
        os.remove(tmp_path)

        if len(gdf) == 0:
            break

        all_parts.append(gdf)

        if len(gdf) < chunk_size:
            break

        offset += chunk_size
        time.sleep(0.2)

    if not all_parts:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    out = pd.concat(all_parts, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")
    return out

def add_ward(df):
    print("Downloading ward layer...")
    wards = download_geojson_layer(WARD_LAYER, out_fields="WARD,NAME,GEOID")

    work = df.copy()
    work["Latitude"] = pd.to_numeric(work["Latitude"], errors="coerce")
    work["Longitude"] = pd.to_numeric(work["Longitude"], errors="coerce")
    work = work.dropna(subset=["Latitude", "Longitude"]).copy()

    points = gpd.GeoDataFrame(
        work,
        geometry=gpd.points_from_xy(work["Longitude"], work["Latitude"]),
        crs="EPSG:4326"
    )

    joined = gpd.sjoin(
        points,
        wards[["WARD", "NAME", "GEOID", "geometry"]],
        how="left",
        predicate="within"
    )

    out = df.copy()
    out["ward"] = None
    out["ward_name"] = None
    out["ward_geoid"] = None

    out.loc[joined.index, "ward"] = joined["WARD"]
    out.loc[joined.index, "ward_name"] = joined["NAME"]
    out.loc[joined.index, "ward_geoid"] = joined["GEOID"]

    return out

def remove_unit_info(street):
    """
    Remove unit/apartment/suite fragments from a street address.
    Examples:
      '123 MAIN ST NW APT 2' -> '123 MAIN ST NW'
      '50 K ST NE #3' -> '50 K ST NE'
      '1010 MASSACHUSETTS AVE NW STE 400' -> '1010 MASSACHUSETTS AVE NW'
    """
    if pd.isna(street):
        return None

    s = clean_whitespace(street).upper()

    patterns = [
        r"\s+\#\s*[\w\-]+$",
        r"\s+(APT|APARTMENT|UNIT|STE|SUITE|FL|FLOOR|RM|ROOM|PH|PENTHOUSE|BSMT|BASEMENT)\s+[\w\-]+.*$",
    ]

    for pat in patterns:
        s = re.sub(pat, "", s, flags=re.I)

    return clean_whitespace(s)


def normalize_street_for_match(street):
    """
    Normalize street address for joining against HistoryQuest ADDRESS.
    Keeps only street portion, removes punctuation, standardizes suffixes.
    """
    if pd.isna(street):
        return None

    s = remove_unit_info(street)
    s = s.upper()

    s = re.sub(r"[^\w\s]", " ", s)

    replacements = {
        r"\bAVENUE\b": "AVE",
        r"\bSTREET\b": "ST",
        r"\bROAD\b": "RD",
        r"\bPLACE\b": "PL",
        r"\bDRIVE\b": "DR",
        r"\bCOURT\b": "CT",
        r"\bCIRCLE\b": "CIR",
        r"\bBOULEVARD\b": "BLVD",
        r"\bTERRACE\b": "TER",
        r"\bLANE\b": "LN",
        r"\bPARKWAY\b": "PKWY",
        r"\bNORTHEAST\b": "NE",
        r"\bNORTHWEST\b": "NW",
        r"\bSOUTHEAST\b": "SE",
        r"\bSOUTHWEST\b": "SW",
    }

    for pattern, repl in replacements.items():
        s = re.sub(pattern, repl, s)

    s = re.sub(r"\s+", " ", s).strip()
    return s


def parse_formatted_address(formatted_address):
    """
    Build clean address parts from 'Formatted Address'.

    Returns:
      street_no_unit
      city
      state
      zip_code
      census_oneline
      match_key

    Assumes DC addresses; defaults to Washington, DC if city/state are missing.
    """
    if pd.isna(formatted_address):
        return {
            "street_no_unit": None,
            "city": None,
            "state": None,
            "zip_code": None,
            "census_oneline": None,
            "match_key": None,
        }

    addr = strip_country(clean_whitespace(formatted_address))
    parts = [p.strip() for p in addr.split(",") if p.strip()]

    # street = first comma-separated part
    street_raw = parts[0] if len(parts) > 0 else None
    street_no_unit = remove_unit_info(street_raw)

    # default DC values
    city = "WASHINGTON"
    state = "DC"

    # try to find zip in full formatted address
    zip_match = re.search(r"\b(\d{5})(?:-\d{4})?\b", addr)
    zip_code = zip_match.group(1) if zip_match else None

    # if city appears explicitly as second segment, keep it
    if len(parts) >= 2:
        maybe_city = parts[1].strip().upper()
        if re.fullmatch(r"[A-Z .'-]+", maybe_city):
            city = maybe_city

    # if state appears in third segment, use it
    state_match = re.search(r"\b([A-Z]{2})\b", addr.upper())
    if state_match:
        maybe_state = state_match.group(1)
        if maybe_state in {"DC", "MD", "VA"}:
            state = maybe_state

    # for Census, either street+zip or street+city+state works
    if zip_code:
        census_oneline = f"{street_no_unit}, {city}, {state} {zip_code}"
    else:
        census_oneline = f"{street_no_unit}, {city}, {state}"

    return {
        "street_no_unit": street_no_unit,
        "city": city,
        "state": state,
        "zip_code": zip_code,
        "census_oneline": census_oneline,
        "match_key": normalize_street_for_match(street_no_unit),
    }


# ---------------------------------------------------------
# CENSUS BATCH GEOCODER
# ---------------------------------------------------------
def build_census_batch_input(unique_addresses_df):
    """
    Build CSV text in the exact Census batch format:
      Unique ID, Street address, City, State, ZIP
    """
    output = io.StringIO()
    writer = csv.writer(output, lineterminator="\n")

    for _, row in unique_addresses_df.iterrows():
        writer.writerow([
            row["addr_id"],
            row["street_no_unit"] if pd.notna(row["street_no_unit"]) else "",
            row["city"] if pd.notna(row["city"]) else "",
            row["state"] if pd.notna(row["state"]) else "",
            row["zip_code"] if pd.notna(row["zip_code"]) else "",
        ])

    return output.getvalue()


def parse_census_batch_response(text):
    """
    Parse Census geographies/addressbatch CSV response.
    Batch geographies returns tract and block code at the end of each row.
    Block group is the first digit of the 4-digit block code.
    """
    rows = []
    reader = csv.reader(io.StringIO(text))

    for parts in reader:
        if not parts:
            continue

        # The exact middle columns can vary, but the final geography
        # fields are typically the last 4:
        # state, county, tract, block
        row_id = parts[0].strip() if len(parts) > 0 else None

        state_fips = parts[-4].strip() if len(parts) >= 4 else None
        county_fips = parts[-3].strip() if len(parts) >= 3 else None
        tract = parts[-2].strip() if len(parts) >= 2 else None
        block = parts[-1].strip() if len(parts) >= 1 else None

        # clean blanks
        state_fips = state_fips if state_fips else None
        county_fips = county_fips if county_fips else None
        tract = tract if tract else None
        block = block if block else None

        block_group = block[0] if block and len(block) >= 1 else None

        rows.append({
            "addr_id": row_id,
            "state_fips": state_fips,
            "county_fips": county_fips,
            "census_tract_geocoder": tract,
            "census_block": block,
            "block_group": block_group,
        })

    return pd.DataFrame(rows)


def run_census_batch(unique_addresses_df, chunk_size=8000):
    """
    Submit addresses in chunks to Census batch geocoder.
    Census batch supports up to 10,000 records per file.
    """
    all_results = []

    total = len(unique_addresses_df)
    if total == 0:
        return pd.DataFrame(columns=[
            "addr_id", "state_fips", "county_fips",
            "census_tract_geocoder", "census_block", "block_group"
        ])

    for start in range(0, total, chunk_size):
        end = min(start + chunk_size, total)
        chunk = unique_addresses_df.iloc[start:end].copy()

        print(f"  Census batch geocoder: {start + 1}-{end} of {total}")

        batch_csv = build_census_batch_input(chunk)

        files = {
            "addressFile": ("addresses.csv", batch_csv.encode("utf-8"), "text/csv")
        }
        data = {
            "benchmark": "Public_AR_Current",
            "vintage": "Current_Current",
        }

        r = requests.post(
            CENSUS_BATCH_URL,
            files=files,
            data=data,
            timeout=300,
            verify=False
        )
        r.raise_for_status()

        result_df = parse_census_batch_response(r.text)
        all_results.append(result_df)

        time.sleep(0.5)

    return pd.concat(all_results, ignore_index=True)


# ---------------------------------------------------------
# HISTORYQUEST / HISTORIC BUILDING INFO
# ---------------------------------------------------------
def fetch_historyquest_attributes():
    """
    Download all historic building attributes needed for address matching.
    Uses returnGeometry=false because we only need attributes.
    """
    all_rows = []
    offset = 0
    page_size = 1000

    while True:
        params = {
            "where": "1=1",
            "outFields": "ADDRESS,DATE_,MAPYEAR,NAME,GIS_ID,HOUSE_KEY,HOUSE_TYPE",
            "returnGeometry": "false",
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }

        r = requests.get(HISTORIC_LAYER, params=params, timeout=120, verify=False)
        r.raise_for_status()
        data = r.json()

        features = data.get("features", [])
        if not features:
            break

        for feat in features:
            attrs = feat.get("attributes", {})
            all_rows.append(attrs)

        print(f"  HistoryQuest rows downloaded: {offset + len(features)}")
        if len(features) < page_size:
            break

        offset += page_size
        time.sleep(0.2)

    hist = pd.DataFrame(all_rows)
    return hist


def prepare_historyquest_lookup():
    hist = fetch_historyquest_attributes()

    # normalize street-only address from HistoryQuest
    hist["match_key"] = hist["ADDRESS"].apply(normalize_street_for_match)

    # DATE_ is epoch milliseconds in many ArcGIS JSON responses
    hist["historic_date"] = pd.to_datetime(hist["DATE_"], unit="ms", errors="coerce")
    hist["historic_year_from_date"] = hist["historic_date"].dt.year
    hist["historic_mapyear_num"] = pd.to_numeric(hist["MAPYEAR"], errors="coerce")

    # prefer DATE_ year, then MAPYEAR
    hist["historic_year_final"] = hist["historic_year_from_date"]
    mask = hist["historic_year_final"].isna() & hist["historic_mapyear_num"].notna()
    hist.loc[mask, "historic_year_final"] = hist.loc[mask, "historic_mapyear_num"]

    # If multiple rows per address, keep the earliest year
    hist = hist.sort_values(
        by=["match_key", "historic_year_final"],
        ascending=[True, True],
        na_position="last"
    )

    hist = hist.dropna(subset=["match_key"]).copy()
    hist = hist.drop_duplicates(subset=["match_key"], keep="first")

    return hist[[
        "match_key", "ADDRESS", "NAME", "GIS_ID", "HOUSE_KEY", "HOUSE_TYPE",
        "historic_date", "historic_year_final"
    ]].rename(columns={
        "ADDRESS": "historic_address",
        "NAME": "historic_name",
        "GIS_ID": "historic_gis_id",
        "HOUSE_KEY": "historic_house_key",
        "HOUSE_TYPE": "historic_house_type",
    })


# ---------------------------------------------------------
# MAIN PIPELINE
# ---------------------------------------------------------
def enrich_file(path, analysis_year=2026):
    print(f"\nProcessing: {path}")
    df = pd.read_csv(path, low_memory=False)

    if "Formatted Address" not in df.columns:
        raise ValueError("Input file must contain 'Formatted Address' column.")

    # Build clean address parts from Formatted Address
    parsed = df["Formatted Address"].apply(parse_formatted_address).apply(pd.Series)
    df = pd.concat([df, parsed], axis=1)

    df = add_ward(df)
    print("  Ward added.")

    # -------------------------
    # 1) Census tract + block group
    # -------------------------
    census_input = df[[
        "street_no_unit", "city", "state", "zip_code",
        "census_oneline"
    ]].copy()

    census_input = census_input.drop_duplicates().reset_index(drop=True)
    census_input["addr_id"] = census_input.index.astype(str)

    census_results = run_census_batch(census_input)

    census_lookup = census_input.merge(census_results, on="addr_id", how="left")

    df = df.merge(
        census_lookup[[
            "street_no_unit", "city", "state", "zip_code", "census_oneline",
            "state_fips", "county_fips",
            "census_tract_geocoder", "census_block", "block_group"
        ]],
        on=["street_no_unit", "city", "state", "zip_code", "census_oneline"],
        how="left"
    )

    # keep existing Census Tract as fallback
    if "Census Tract" in df.columns:
        df["census_tract_final"] = df["census_tract_geocoder"]
        mask = df["census_tract_final"].isna() & df["Census Tract"].notna()
        df.loc[mask, "census_tract_final"] = df.loc[mask, "Census Tract"]
    else:
        df["census_tract_final"] = df["census_tract_geocoder"]

    # -------------------------
    # 2) Building age from HistoryQuest
    # -------------------------
    hist_lookup = prepare_historyquest_lookup()

    df = df.merge(
        hist_lookup,
        on="match_key",
        how="left"
    )

    df["building_age"] = df["historic_year_final"].apply(
        lambda y: analysis_year - y if pd.notna(y) else None
    )

    # -------------------------
    # Save
    # -------------------------
    output_path = path.replace(".csv", "_formatted_address_enriched.csv")
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    print("\nMatch summary:")
    print("  Tract from Census geocoder:", df["census_tract_geocoder"].notna().sum(), "of", len(df))
    print("  Final tract (with fallback):", df["census_tract_final"].notna().sum(), "of", len(df))
    print("  Block group:", df["block_group"].notna().sum(), "of", len(df))
    print("  Building age:", df["building_age"].notna().sum(), "of", len(df))

    return df


# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------
if __name__ == "__main__":
    for file_path in INPUT_FILES:
        enrich_file(file_path, analysis_year=2026)


Processing: final_merged_5_geocoded.csv
  Ward added.
  Census batch geocoder: 1-862 of 862
  HistoryQuest rows downloaded: 1000
  HistoryQuest rows downloaded: 2000
  HistoryQuest rows downloaded: 3000
  HistoryQuest rows downloaded: 4000
  HistoryQuest rows downloaded: 5000
  HistoryQuest rows downloaded: 6000
  HistoryQuest rows downloaded: 7000
  HistoryQuest rows downloaded: 8000
  HistoryQuest rows downloaded: 9000
  HistoryQuest rows downloaded: 10000
  HistoryQuest rows downloaded: 11000
  HistoryQuest rows downloaded: 12000
  HistoryQuest rows downloaded: 13000
  HistoryQuest rows downloaded: 14000
  HistoryQuest rows downloaded: 15000
  HistoryQuest rows downloaded: 16000
  HistoryQuest rows downloaded: 17000
  HistoryQuest rows downloaded: 18000
  HistoryQuest rows downloaded: 19000
  HistoryQuest rows downloaded: 20000
  HistoryQuest rows downloaded: 21000
  HistoryQuest rows downloaded: 22000
  HistoryQuest rows downloaded: 23000
  HistoryQuest rows downloaded: 24000
  Hi